In [0]:


# COMMAND ----------
# Cria o schema da camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# COMMAND ----------
# Caminho dos arquivos no Volume
volume_path = "/Volumes/csvs/default/olist_raw"

# COMMAND ----------
# Valida se os arquivos estão disponíveis
display(dbutils.fs.ls(volume_path))

# COMMAND ----------
# Importações
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import requests

# COMMAND ----------
# Função para carregar CSV na Bronze com timestamp de ingestão
def carregar_csv_para_bronze(nome_arquivo, nome_tabela):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{volume_path}/{nome_arquivo}")
        .withColumn("timestamp_ingestion", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

    print(f"Tabela {nome_tabela} criada com sucesso.")

# COMMAND ----------
# Mapeamento dos arquivos para tabelas Bronze
arquivos_tabelas = {
    "olist_customers_dataset.csv": "bronze.tb_customers",
    "olist_geolocation_dataset.csv": "bronze.tb_geolocalizacao",
    "olist_order_items_dataset.csv": "bronze.tb_order_items",
    "olist_order_payments_dataset.csv": "bronze.tb_order_payments",
    "olist_order_reviews_dataset.csv": "bronze.tb_order_reviews",
    "olist_orders_dataset.csv": "bronze.tb_orders",
    "olist_products_dataset.csv": "bronze.tb_products",
    "olist_sellers_dataset.csv": "bronze.tb_sellers",
    "product_category_name_translation.csv": "bronze.tb_product_category_name_translation"
}

# COMMAND ----------
# Ingestão de todos os CSVs na camada Bronze
for arquivo, tabela in arquivos_tabelas.items():
    carregar_csv_para_bronze(arquivo, tabela)

# COMMAND ----------
# Widgets para definir o período da API do dólar
dbutils.widgets.text("data_inicio", "01-01-2017")
dbutils.widgets.text("data_fim", "12-31-2018")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print("Data início:", data_inicio)
print("Data fim:", data_fim)

# COMMAND ----------
# Consome a API do Banco Central para obter a cotação do dólar
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Erro ao consultar API do Banco Central. Status code: {response.status_code}")

json_data = response.json()

if "value" not in json_data:
    raise Exception("Resposta da API não contém a chave 'value'.")

dados_dolar = json_data["value"]

print(f"Quantidade de registros retornados pela API: {len(dados_dolar)}")

# COMMAND ----------
# Cria DataFrame da cotação do dólar
schema_dolar = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

df_dolar = (
    spark.createDataFrame(dados_dolar, schema=schema_dolar)
    .withColumn("timestamp_ingestion", current_timestamp())
)

# COMMAND ----------
# Salva a cotação do dólar na Bronze
(
    df_dolar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

print("Tabela bronze.tb_cotacao_dolar criada com sucesso.")

# COMMAND ----------
# Lista as tabelas criadas na camada Bronze
spark.sql("SHOW TABLES IN bronze").display()

# COMMAND ----------
# Conferência visual de algumas tabelas
spark.table("bronze.tb_customers").display()
spark.table("bronze.tb_orders").display()
spark.table("bronze.tb_cotacao_dolar").display()
